# 01. 데이터 탐색

3-class 자세 분류 데이터의 분포, 균형, 샘플 시각화

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

RAW_ROOT = Path('../data/raw')
SPLITS_CSV = Path('../data/splits/splits.csv')
CLASSES = ['neutral', 'forward_head', 'slouching']

In [ ]:
counts = {}
for cls in CLASSES:
    d = RAW_ROOT / cls
    if d.exists():
        imgs = [f for f in d.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png'}]
        counts[cls] = len(imgs)
    else:
        counts[cls] = 0

print('=== 클래스별 이미지 수 ===')
for k, v in counts.items():
    print(f'  {k}: {v}장')
print(f'  합계: {sum(counts.values())}장')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.keys(), counts.values(), color=['#2ecc71','#e74c3c','#3498db'])
ax.bar_label(bars)
ax.set_title('클래스별 이미지 수')
ax.set_ylabel('이미지 수')
plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/class_distribution.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for row, cls in enumerate(CLASSES):
    cls_dir = RAW_ROOT / cls
    if not cls_dir.exists():
        for ax in axes[row]:
            ax.axis('off')
        continue
    imgs = sorted(cls_dir.glob('*.jpg'))[:5] + sorted(cls_dir.glob('*.png'))[:5]
    imgs = imgs[:5]
    for col, img_path in enumerate(imgs):
        img = Image.open(img_path).convert('RGB')
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_title(cls, fontsize=12, fontweight='bold')
    for col in range(len(imgs), 5):
        axes[row][col].axis('off')
plt.suptitle('클래스별 샘플 이미지', fontsize=14)
plt.tight_layout()
plt.savefig('../results/figures/sample_images.png', dpi=150)
plt.show()

In [ ]:
if SPLITS_CSV.exists():
    df = pd.read_csv(SPLITS_CSV)
    pivot = df.groupby(['split', 'label']).size().unstack(fill_value=0)
    print(pivot)
    pivot.plot(kind='bar', figsize=(10, 5), title='Split별 클래스 분포')
    plt.tight_layout()
    plt.savefig('../results/figures/split_distribution.png', dpi=150)
    plt.show()
else:
    print('splits.csv 없음. scripts/build_splits.py 먼저 실행하세요.')

In [ ]:
widths, heights = [], []
for cls in CLASSES:
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        for p in (RAW_ROOT / cls).glob(ext):
            try:
                w, h = Image.open(p).size
                widths.append(w)
                heights.append(h)
            except:
                pass

if widths:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(widths, bins=30, color='steelblue')
    axes[0].set_title('Width 분포')
    axes[1].hist(heights, bins=30, color='salmon')
    axes[1].set_title('Height 분포')
    plt.tight_layout()
    plt.savefig('../results/figures/resolution_dist.png', dpi=150)
    plt.show()
    print(f'Width: {np.mean(widths):.0f}±{np.std(widths):.0f}, Height: {np.mean(heights):.0f}±{np.std(heights):.0f}')